# Recomendación híbrida: contenido + colaborativo

Este notebook combina la factorización matricial de referencia y TF-IDF mediante puntuaciones normalizadas por usuario. Los pesos se seleccionan con validación; prueba se reserva para el resultado del peso elegido. Todos los modelos usan los candidatos v2 persistidos, por lo que comparan usuarios, exclusiones y relevancia idénticos.

In [1]:
from pathlib import Path
import json, sys
import joblib, numpy as np, pandas as pd, scipy.sparse as sp
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'ml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from ml.recommendation import ranking_metrics, minmax_by_user, book_ids_for_indices
PROCESSED, ML = ROOT / 'data' / 'processed', ROOT / 'data' / 'ml'
CONTENT = ROOT / 'artifacts' / 'content_tfidf'; ARTIFACTS = ROOT / 'artifacts' / 'hybrid'; ARTIFACTS.mkdir(parents=True, exist_ok=True)
TOP_K = 10
valid = pd.read_parquet(CONTENT / 'valid_candidate_scores.parquet'); test = pd.read_parquet(CONTENT / 'test_candidate_scores.parquet')
book_mapping = pd.read_parquet(ML / 'book_mapping.parquet'); books = pd.read_parquet(PROCESSED / 'books.parquet')[['book_id', 'title']]


## 1. Protocolo y normalización
El candidato relevante es un rating retenido de al menos cuatro estrellas. Cada usuario recibe el mismo relevante y 1,000 ítems no vistos. La normalización min-max se calcula únicamente dentro de los candidatos y puntuaciones del usuario, sin usar etiquetas de validación o prueba.

In [2]:
def normalized(frame):
    result = minmax_by_user(frame, 'collaborative_score', 'collaborative_norm')
    return minmax_by_user(result, 'content_score', 'content_norm')
valid_norm, test_norm = normalized(valid), normalized(test)
assert valid_norm[['user_idx', 'book_idx', 'relevant']].equals(valid[['user_idx', 'book_idx', 'relevant']])
print(f'Usuarios/candidatos validación: {valid_norm.user_idx.nunique():,}/{len(valid_norm):,}')


Usuarios/candidatos validación: 5,000/5,005,000


## 2. Selección de pesos con validación
Se prueban pesos de contenido entre 0 y 1. `score_híbrido = w * contenido_normalizado + (1 - w) * colaborativo_normalizado`. Esta búsqueda no usa la prueba ni reentrena los modelos individuales.

In [3]:
weight_rows = []
for weight in np.linspace(0, 1, 11):
    trial = valid_norm.copy(); trial['hybrid_score'] = weight * trial.content_norm + (1 - weight) * trial.collaborative_norm
    _, metrics = ranking_metrics(trial, 'hybrid_score', TOP_K)
    weight_rows.append({'content_weight': float(weight), 'collaborative_weight': float(1 - weight), **metrics})
weight_results = pd.DataFrame(weight_rows).sort_values(['ndcg_at_k', 'content_weight'], ascending=[False, False])
display(weight_results)
best = weight_results.iloc[0].to_dict(); content_weight = float(best['content_weight'])
display(Markdown(f"**Interpretación.** Se selecciona `w={content_weight:.1f}` por NDCG@10 de validación. Si el híbrido no supera al mejor individual, se conserva ese resultado: una combinación ponderada no garantiza complementariedad."))


,content_weight,collaborative_weight,precision_at_k,recall_at_k,ndcg_at_k
6,0.6,0.4,0.04324,0.4324,0.304977
5,0.5,0.5,0.04250,0.4250,0.304562
7,0.7,0.3,0.04348,0.4348,0.303097
8,0.8,0.2,0.04300,0.4300,0.298588
4,0.4,0.6,0.04052,0.4052,0.292927
9,0.9,0.1,0.04184,0.4184,0.291397
10,1.0,0.0,0.04106,0.4106,0.285293
3,0.3,0.7,0.03574,0.3574,0.253524
2,0.2,0.8,0.02796,0.2796,0.184729
1,0.1,0.9,0.02004,0.2004,0.124457


**Interpretación.** Se selecciona `w=0.6` por NDCG@10 de validación. Si el híbrido no supera al mejor individual, se conserva ese resultado: una combinación ponderada no garantiza complementariedad.

## 3. Comparación final sobre candidatos idénticos
La prueba se utiliza solo ahora, con el peso fijado. Popularidad, colaborativo, contenido e híbrido se ordenan sobre exactamente las mismas filas de candidatos ya persistidas.

In [4]:
def evaluate(frame, partition):
    hybrid = frame.copy(); hybrid['hybrid_score'] = content_weight * hybrid.content_norm + (1 - content_weight) * hybrid.collaborative_norm
    rows = []
    for label, column in [('Popularidad', 'popularity_score'), ('Colaborativo', 'collaborative_score'), ('Contenido', 'content_score'), ('Híbrido', 'hybrid_score')]:
        detail, metrics = ranking_metrics(hybrid, column, TOP_K)
        rows.append({'partición': partition, 'modelo': label, 'usuarios': len(detail), **metrics})
    return hybrid, pd.DataFrame(rows)
valid_hybrid, valid_results = evaluate(valid_norm, 'validación'); test_hybrid, test_results = evaluate(test_norm, 'prueba')
results = pd.concat([valid_results, test_results], ignore_index=True); display(results)
results.to_json(ARTIFACTS / 'metrics.json', orient='records', indent=2, force_ascii=False)
hybrid_config = {'content_weight': content_weight, 'collaborative_weight': 1 - content_weight, 'normalization': 'minmax_per_user_candidate_set', 'selection_metric': 'validation_ndcg_at_10', 'candidate_protocol': 'one_relevant_plus_1000_unseen_train_items'}
(ARTIFACTS / 'config.json').write_text(json.dumps(hybrid_config, indent=2), encoding='utf-8')
test_hybrid.to_parquet(ARTIFACTS / 'test_candidate_scores.parquet', index=False)


,partición,modelo,usuarios,precision_at_k,recall_at_k,ndcg_at_k
0,validación,Popularidad,5000,0.04034,0.4034,0.265721
1,validación,Colaborativo,5000,0.01450,0.1450,0.087826
2,validación,Contenido,5000,0.04106,0.4106,0.285293
3,validación,Híbrido,5000,0.04324,0.4324,0.304977
4,prueba,Popularidad,5000,0.03732,0.3732,0.245294
5,prueba,Colaborativo,5000,0.01372,0.1372,0.079582
6,prueba,Contenido,5000,0.03142,0.3142,0.209847
7,prueba,Híbrido,5000,0.03510,0.3510,0.232719


## 4. Ejemplos e inicio en frío
Para un usuario existente se excluyen los ítems de entrenamiento por construcción de candidatos. Para uno nuevo no existe embedding colaborativo: se usa un perfil explícito de texto, autores o géneros como consulta TF-IDF. Este fallback no pretende reemplazar el filtrado colaborativo.

In [5]:
example_user = int(test_hybrid.user_idx.iloc[0]); example = test_hybrid[test_hybrid.user_idx.eq(example_user)].nlargest(10, 'hybrid_score')
example_ids = book_ids_for_indices(book_mapping, example.book_idx.to_numpy(dtype=np.int64))
display(pd.DataFrame({'book_id': example_ids, 'hybrid_score': example.hybrid_score.to_numpy()}).merge(books, on='book_id', how='left'))
vectorizer = joblib.load(CONTENT / 'tfidf_vectorizer.joblib'); matrix = sp.load_npz(CONTENT / 'book_tfidf.npz'); content_map = pd.read_parquet(CONTENT / 'content_book_mapping.parquet')
NEW_USER_PREFERENCES = 'poetry poems verse literature'
profile = vectorizer.transform([NEW_USER_PREFERENCES]); scores = (profile @ matrix.T).toarray().ravel(); rows = np.argsort(-scores)[:10]
cold = content_map.iloc[rows][['book_id']].merge(books, on='book_id', how='left'); display(cold)
display(Markdown('**Interpretación.** El ejemplo de usuario nuevo solo usa preferencias declaradas. Un usuario o libro nuevo no puede recibir una puntuación colaborativa hasta que exista historial entrenable; el híbrido debe degradar de forma explícita a contenido o popularidad.'))


,book_id,hybrid_score,title
0,26891650,0.844392,Saga #27
1,15893436,0.788893,Hideyuki Kikuchi's Vampire Hunter D Volume 7
2,1298878,0.786649,God Child 1
3,2142254,0.773944,Dolis: Volume 1
4,460584,0.734470,Ohikkoshi
5,22469996,0.732935,Mýty 4: Pochod dřevěných vojáčků
6,3259434,0.690292,"Loveless, Volume 8"
7,9367853,0.684408,"Fables (The Deluxe Edition, #3)"
8,9968822,0.683056,"Transmetropolitan, Vol. 10: One More Time"
9,22291145,0.652995,Patricia Briggs' Mercy Thompson: Homecoming #1...


,book_id,title
0,1488078,Contemporary Indonesian Poetry: Poems In Bahas...
1,15888643,I'm Not a Poet But I Write Poetry: Poems from ...
2,30053396,Honeycomb Poems
3,20423140,"T. S. Eliot: Collection of Poetry, Poems, and ..."
4,2424639,Erotic Verse
5,15724047,Lady Business - A Celebration of Lesbian Poetry
6,5634310,Neck-Verse
7,240026,Against Love Poetry: Poems
8,1951226,Poems for a Small Planet: Contemporary America...
9,11202210,The Captain Poetry Poems Complete


**Interpretación.** El ejemplo de usuario nuevo solo usa preferencias declaradas. Un usuario o libro nuevo no puede recibir una puntuación colaborativa hasta que exista historial entrenable; el híbrido debe degradar de forma explícita a contenido o popularidad.

## 5. Limitaciones
- Los negativos son candidatos no observados, no rechazos confirmados.
- El muestreo de 1,000 candidatos no equivale al catálogo completo; su objetivo es comparación reproducible y coste controlado.
- La poda iterativa previa a la partición reduce usuarios/ítems de cola larga y puede sobrestimar cobertura respecto a futuros usuarios nuevos.
- `date_added_utc` representa incorporación a Goodreads, no lectura; el protocolo temporal es una aproximación.
- Si popularidad supera al híbrido, no debe sustituirse el baseline sin una mejora validada.